# **Import Utilities**

In [1]:
# import pyarrow as pa
# import pyarrow.parquet as pq
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pandas as pd
import json

import sys
sys.path.append('../')  # Add the parent directory where 'utils' is to the Python path
from utils.statistics_plots_analysis_utils import *
from utils.data_setup import load_existing_responses

TASK_SUBSET = "1000"           # "all" | "1000" | "50"
PROMPTING_TYPE = "zero_shot"    # "zero_shot" | "few_shot"
STAGE = "critiques"              # "ratings" or "critiques"
NUM_TRIALS = 1

MODEL_SHORT = {
    'GPT-5': 'gpt5',
    'Gemini-2.5': 'gemini2_5pro',
    'Claude-4': 'claude4',
    'Qwen-3': 'qwen3_vl',
}

In [ ]:
SELECTED_TASKS = f"{TASK_SUBSET}_screens" if TASK_SUBSET != "all" else "all_tasks"

dataset_dir = Path("../../data/dataset/cleaned_dataset")
uicrit_data_file = dataset_dir / "uicrit_notna_deduped.parquet"
base64_screens_file = Path(dataset_dir) / "base64_screens.parquet"

results_dir = Path(f"../results/{STAGE}/parquet")

results_file_base = f"{STAGE}-{{prompting_type}}-{SELECTED_TASKS}-{{model_short}}-responses.parquet"

selected_models = [
    "GPT-5", 
    # "Gemini-2.5", 
    # "Claude-4", 
    # "Qwen-3"
    ]

# **Load Data**

Load Experts Data

In [3]:
# load experts data
experts_df = pd.read_parquet(uicrit_data_file, columns=['screen_task_id', 'screen_id', 'parsed_comments'])

print('experts_df shape:', experts_df.shape)
experts_df.head(1)

experts_df shape: (2957, 3)


,screen_task_id,screen_id,parsed_comments
0,15_T01,15,"[{'bounding_box': [0.1559446, 0.05137866, 0.40..."


**Load and Merge Model Responses to Experts**

In [4]:
def load_and_merge_models(prompting_type, selected_models, results_file_base):
    model_results_files = {
        model: results_dir / results_file_base.format(model_short=MODEL_SHORT[model], prompting_type=prompting_type)
        for model in selected_models
    }

    model_dfs = {}

    for model_name, path in model_results_files.items():
        df = load_existing_responses(
            results_file_path=path,
            uicrit_file=uicrit_data_file,
            num_trials=NUM_TRIALS,
            stage=STAGE,
        )

        if "critiques" in df.columns:
            df["critiques"] = df["critiques"].apply(
                lambda x: json.loads(x) if isinstance(x, str) else x
            )

        merged_df = pd.merge(df, experts_df, on=["screen_task_id", "screen_id"])
        model_dfs[model_name] = merged_df

        print(f"{prompting_type} | {model_name} shape: {merged_df.shape} | N/A count:", merged_df.isna().sum().sum())

    return model_dfs

model_dfs_zs = load_and_merge_models("zero_shot", selected_models, results_file_base)
# model_dfs_fs = load_and_merge_models("few_shot", selected_models, results_file_base)

zero_shot | GPT-5 shape: (1000, 5) | N/A count: 0


In [5]:
responses_df = model_dfs_zs['GPT-5']
responses_df

,screen_task_id,screen_id,task,critiques,parsed_comments
0,15_T01,15,Plan and Start Full Body Workouts,"[{'expected_standard': 'A clear, prominent pri...","[{'bounding_box': [0.1559446, 0.05137866, 0.40..."
1,28_T01,28,Enter details to Sing In to Scotiabank.,[{'expected_standard': 'Input fields have pers...,"[{'bounding_box': [0.66785016, 0.13043726, 0.9..."
2,67_T01,67,Upload the Mississippi River Delta image to Ea...,[{'expected_standard': 'Provide a clearly reco...,"[{'bounding_box': [0.027119, 0.01355937, 0.120..."
3,190_T01,190,Click the 'Chat' button to initiate a conversa...,[{'expected_standard': 'Primary actions should...,"[{'bounding_box': [0.76881927, 0.09380508, 1.0..."
4,193_T01,193,Search and explore friends,[{'expected_standard': 'Primary actions and in...,"[{'bounding_box': [0.07768441, 0.31092437, 0.9..."
...,...,...,...,...,...
995,71737_T01,71737,Explore selected options from the menu,[{'expected_standard': 'Navigation items shoul...,"[{'bounding_box': [0.05378151, 0.60504202, 0.1..."
996,71747_T01,71747,Choose option to check for Psychiatric help on...,[{'expected_standard': 'Navigation should pres...,"[{'bounding_box': [0.03970981, 0.25302606, 0.9..."
997,72017_T01,72017,Browse options to find your match,[{'expected_standard': 'Primary tasks should b...,"[{'bounding_box': [0.10982702, 0.2715625, 0.24..."
998,72147_T01,72147,Enter details for new member registration,[{'expected_standard': 'Each form field has a ...,"[{'bounding_box': [0.0030546, 0.62199313, 0.40..."


In [6]:
# keep in experts_df rows that are in any model_dfs_zs
experts_df_filtered = experts_df[
    experts_df.apply(
        lambda row: any(
            row['screen_task_id'] in model_dfs_zs[model]['screen_task_id'].values and
            row['screen_id'] in model_dfs_zs[model]['screen_id'].values
            for model in model_dfs_zs
        ),
        axis=1
    )
]
print('experts_df_filtered shape:', experts_df_filtered.shape)
experts_df_filtered.head(1)

experts_df_filtered shape: (1000, 3)


,screen_task_id,screen_id,parsed_comments
0,15_T01,15,"[{'bounding_box': [0.1559446, 0.05137866, 0.40..."


Make it ONE dataframe

# **Compare Comments**

In [ ]:
list_A = [{"expected_standard": "The selected workout should present a clear, prominent primary action (e.g., a Start button) labeled with a verb and provide immediate feedback when invoked.",
      "observed_issue": "The Full Body item only shows a small checkmark on the right with no explicit Start action or indication that tapping the row will begin the workout, leaving users to guess how to start.",
      "suggested_fix": "When Full Body is selected, surface a prominent \"Start Full Body\" button (e.g., sticky at the bottom or inline in the row) and make the entire row tappable to start, with immediate transition/feedback.",
      "guideline_reference": "Nielsen Norman Heuristics — Recognition rather than recall; Visibility of system status. Apple Human Interface Guidelines — Buttons: Use verbs in titles and emphasize primary actions."},
    {"expected_standard": "Floating controls should respect safe areas and never obscure primary content or controls users need to read or tap.",
      "observed_issue": "The teal floating + button overlaps the Lower Body row, partially covering its lock/info controls and likely blocking taps.",
      "suggested_fix": "Add bottom padding/inset to the list equal to the FAB size, reposition or auto-hide the FAB on scroll, or move it where it won’t cover list content.",
      "guideline_reference": "Apple Human Interface Guidelines — Layout and Safe Areas: Don’t let controls obscure content. Nielsen Norman Heuristics — Error prevention."},
    {"expected_standard": "Text should maintain sufficient color contrast against its background to remain legible in varied lighting conditions.",
      "observed_issue": "The \"Get Started\" section label and descriptive copy are rendered in a pale orange on white, resulting in low contrast and reduced readability.",
      "suggested_fix": "Use a higher-contrast text color that meets accessibility contrast guidelines (e.g., dark gray/black for body and section labels) and reserve the accent color for emphasis or interactive elements.",
      "guideline_reference": "Apple Human Interface Guidelines — Typography and Color: Ensure adequate contrast for readability. CrowdCrit Visual Design Critiques — Contrast and Legibility."},
    {"expected_standard": "Icons that aren’t universally recognized should include text labels or clear contextual cues so users don’t have to infer meaning.",
      "observed_issue": "Right-aligned icons (checkmark, lock, info) are unlabeled. The checkmark’s meaning (selected, started, or completed) is ambiguous, forcing interpretation.",
      "suggested_fix": "Replace the checkmark with a labeled CTA (e.g., \"Start\"), add a small \"Locked\"/price label next to the lock, or pair icons with short labels. Ensure consistent icon semantics across the list.",
      "guideline_reference": "Nielsen Norman Heuristics — Recognition rather than recall; Consistency and standards. Apple Human Interface Guidelines — Icons: Use familiar symbols and provide text labels for clarity."},
    {"expected_standard": "Unavailable or paywalled items should be clearly identified and explain how to gain access to prevent dead-ends.",
      "observed_issue": "Locked workouts look visually similar to available ones and provide no inline explanation (e.g., price or requirement), likely leading to taps that end at a paywall.",
      "suggested_fix": "Visually de-emphasize locked rows (reduced opacity, disabled state), disable the primary tap, and include a concise message like \"Premium — Subscribe to unlock\" with a clear CTA.",
      "guideline_reference": "Nielsen Norman Heuristics — Error prevention; Help users recognize, diagnose, and recover from errors. Apple Human Interface Guidelines — Feedback: Clearly communicate availability and status."},
    {"expected_standard": "Interactive controls should have comfortable touch targets with sufficient spacing from edges and neighboring controls.",
      "observed_issue": "The small info and lock icons are tightly packed near the right edge, making them hard to tap accurately, especially one-handed.",
      "suggested_fix": "Increase each control’s tappable area to at least 44pt x 44pt (iOS) / 48dp (Android), add padding between icons, and make the entire row the primary tap target.",
      "guideline_reference": "Apple Human Interface Guidelines — Touch Targets: Minimum 44pt x 44pt. Nielsen Norman Heuristics — Error prevention."}
  ]

list_B = [{'expected_standard': 'Primary actions should be explicit and discoverable with verb-labeled controls (e.g., a Start button) so users know exactly how to initiate a workout.',
  'observed_issue': 'The Full Body item shows a checkmark but no clear Start control; it’s unclear whether tapping the row starts the workout or merely selects it.',
  'suggested_fix': 'Add a prominent, labeled "Start Full Body" button (or a play button with the label "Start") visible when Full Body is selected, and make the entire row clearly tappable with feedback that the workout is starting.',
  'guideline_reference': 'Nielsen Norman: Visibility of System Status; Recognition Rather Than Recall. Apple Human Interface Guidelines: Use verbs in button titles and make primary actions obvious.'},
 {'expected_standard': 'Floating controls must not obscure readable content or tappable elements and should respect safe areas.',
  'observed_issue': 'The + floating action button overlaps the Lower Body row, partially covering text and the right-side controls.',
  'suggested_fix': 'Reposition the FAB to a bottom app bar, increase its inset from list items, or hide it on scroll so it never covers list content.',
  'guideline_reference': 'Apple Human Interface Guidelines: Respect safe areas so content isn’t obscured. Nielsen Norman: Error Prevention.'},
 {'expected_standard': 'Interactive elements should have comfortable hit targets (at least 44x44 pt) with adequate spacing to prevent mistaps.',
  'observed_issue': 'The lock and info icons on the right of each row appear small and closely spaced, making precise tapping difficult.',
  'suggested_fix': 'Increase the tappable area of each icon to at least 44x44 pt with sufficient padding or move secondary actions into a details screen.',
  'guideline_reference': 'Apple Human Interface Guidelines: Touch targets should be at least 44x44 pt. Nielsen Norman: Error Prevention.'},
 {'expected_standard': 'Icon positions and meanings should be consistent so the same location conveys the same type of information or action across items.',
  'observed_issue': 'The right-side icon slot shows a lock for some items and a checkmark for Full Body, mixing "locked" and "selected" states in the same position without labels.',
  'suggested_fix': 'Use a consistent status treatment (e.g., a labeled "Locked" badge) and move the selection indicator to a different, consistent location (e.g., leading checkmark or highlighted row).',
  'guideline_reference': 'Nielsen Norman: Consistency and Standards. Apple Human Interface Guidelines: Be consistent with the placement and meaning of controls and labels.'},
 {'expected_standard': 'Text should have sufficient color contrast against its background for easy reading and scanning.',
  'observed_issue': 'The "Get Started" heading and secondary descriptions use a light orange on white, resulting in low contrast and reduced legibility.',
  'suggested_fix': 'Use a darker text color (e.g., system label colors) that meets at least WCAG AA contrast and avoid relying on low-contrast color to denote hierarchy.',
  'guideline_reference': 'Apple Human Interface Guidelines: Ensure text is legible and provides sufficient contrast. Nielsen Norman: Visibility of System Status.'}]

list_C = [{'expected_standard': 'After choosing a workout, the next step to begin should be obvious and immediately available via a prominent, clearly labeled primary action (e.g., Start Full Body).',
   'observed_issue': 'The Full Body row only shows a small checkmark; there is no clear Start action or visible next step to begin the workout, creating uncertainty about how to proceed.',
   'suggested_fix': 'Add a prominent Start Full Body button (e.g., a sticky bottom bar or trailing button on the selected row) and make tapping the row itself start or open a start screen with clear feedback.',
   'guideline_reference': 'Nielsen Norman 10 Heuristics: Visibility of system status; Recognition rather than recall; User control and freedom. Apple Human Interface Guidelines: Buttons — Emphasize primary actions; Feedback — Provide clear feedback for user actions.'},
  {'expected_standard': 'Floating controls should not obscure content or interactive elements; lists should include bottom padding/safe-area so all items remain readable and tappable.',
   'observed_issue': 'The floating + button overlaps the Lower Body list item and its lock icon, obstructing content and potential taps.',
   'suggested_fix': 'Add bottom inset/padding to the list, reposition the FAB away from list content, or auto-hide it while scrolling so no items are covered.',
   'guideline_reference': 'Apple Human Interface Guidelines: Layout — Respect safe areas and avoid obscuring content. Nielsen Norman 10 Heuristics: Error prevention.'},
  {'expected_standard': 'Text and essential icons must meet accessible contrast (e.g., WCAG AA ~4.5:1) to ensure readability in varied conditions.',
   'observed_issue': 'Light orange text (e.g., section headers and descriptions) on a white background and pale teal icons exhibit low contrast, reducing readability.',
   'suggested_fix': 'Use darker text colors from the platform’s standard palette and adjust icon tints to achieve at least AA contrast across backgrounds.',
   'guideline_reference': 'Apple Human Interface Guidelines: Color and Contrast — Ensure sufficient contrast for readability. CrowdCrit Visual Design Critiques: Insufficient contrast reduces legibility.'},
  {'expected_standard': 'Icons should be easily understood or accompanied by text labels so users can recognize actions without guessing.',
   'observed_issue': 'Trailing icons (i, lock, check) appear without labels, making it unclear whether they open details, indicate status, or perform an action.',
   'suggested_fix': 'Add short labels (e.g., Details, Locked, Selected) or convert to labeled buttons; ensure accessible names so assistive technologies announce their purpose.',
   'guideline_reference': 'Nielsen Norman 10 Heuristics: Recognition rather than recall; Consistency and standards. Apple Human Interface Guidelines: Icons — Use familiar icons and include text when necessary.'},
  {'expected_standard': 'Interactive controls should have hit areas of at least 44x44 pt with adequate padding from edges and nearby controls.',
   'observed_issue': 'The small info and lock icons on the right edge appear smaller than 44x44 pt and closely packed, making them hard to tap accurately.',
   'suggested_fix': 'Increase the tappable area of trailing icons to 44x44 pt and add spacing from the screen edge and between icons.',
   'guideline_reference': 'Apple Human Interface Guidelines: Touch Targets — Make controls at least 44pt x 44pt.'},
  {'expected_standard': 'Status indicators should clearly communicate meaning with explicit labels or well-understood patterns and consistent color semantics.',
   'observed_issue': 'A small teal checkmark beside Full Body is unlabeled and similar in color to other icons, making it ambiguous whether it means selected, completed, or active.',
   'suggested_fix': 'Pair the checkmark with text such as Selected or Current, or replace it with a clear state chip; use consistent colors to differentiate state from actions.',
   'guideline_reference': 'Nielsen Norman 10 Heuristics: Visibility of system status; Consistency and standards. Apple Human Interface Guidelines: Feedback — Communicate status and changes clearly; Color — Use color consistently to convey meaning.'},
  {'expected_standard': 'When content is locked, the interface should explain why and provide a clear, actionable path to unlock it (e.g., upgrade, subscribe).',
   'observed_issue': 'Items under Plans display a lock icon with no explanatory text or CTA, leaving users unsure how to access them when planning workouts.',
   'suggested_fix': 'Show a short message like Premium plan—Upgrade to unlock with a visible Upgrade/Subscribe button; tapping a locked item should present an upgrade sheet with details.',
   'guideline_reference': 'Nielsen Norman 10 Heuristics: Visibility of system status; Help and documentation; Match between system and the real world. Apple Human Interface Guidelines: Empty States and Feedback — Provide informative, actionable messaging.'}]

observed_A = [c["observed_issue"] for c in list_A]
observed_B = [c["observed_issue"] for c in list_B]
observed_C = [c["observed_issue"] for c in list_C]
uicrit_df = pd.read_parquet(paths["uicrit_data"])
list_expert = uicrit_df.loc[0, 'comments']
list_expert

array(['Comment 1\nThe expected standard is that the text’s visual treatment and formatting should make it easy to understand. In the current design, the text (Workouts) is small even when it is heading. To fix this, increase font size and weight to make it  look like a heading \n\nBounding Box: [0.1559446, 0.05137866, 0.40545597, 0.09649163]',
       'Comment 2\nThe expected standard is that the text and background colors used in the design should be complementary and easy to read. In the current design, text (Plans) is in light red color on white background  which is not making a good contrast. To fix this, change colors to be more complementary to each other (change texts to dark colors) to make it a good contrast and easier to read.\n\nBounding Box: [0.02896114, 0.15037657, 0.1559446, 0.18045189]',
       'Comment 3\nThe expected standard is that the design should make the most important information visually dominant. \nIn the current design, the back button size is small.\nTo fix 

In [ ]:
for i, item in enumerate(observed_A):
    if i > 2 and i != 5:
        print(f"A{i}: {item}")


A3: Right-aligned icons (checkmark, lock, info) are unlabeled. The checkmark’s meaning (selected, started, or completed) is ambiguous, forcing interpretation.
A4: Locked workouts look visually similar to available ones and provide no inline explanation (e.g., price or requirement), likely leading to taps that end at a paywall.


### Comparing Methods

#### TF-IDF

In [ ]:
from typing import List, Dict, Any, Tuple
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def compare_critique_lists_tfidf(
    list_A: List[Dict[str, Any]],
    list_B: List[Dict[str, Any]],
    field: str = "observed_issue",
    sim_threshold: float = 0.3,
):
    """
    Compare two lists of critiques using TF-IDF + cosine similarity
    over a selected text field (default: 'observed_issue').

    Returns a dict with:
      - similarity_matrix: np.ndarray [len(A) x len(B)]
      - matches: List[(idx_A, idx_B, similarity)]
      - A_only: critiques in A with no match above threshold
      - B_only: critiques in B with no match above threshold
    """

    # 1) Extract texts
    A_texts = [c.get(field, "") for c in list_A]
    B_texts = [c.get(field, "") for c in list_B]

    if not A_texts or not B_texts:
        return {
            "similarity_matrix": None,
            "matches": [],
            "A_only": list_A,
            "B_only": list_B,
        }

    # 2) Build TF-IDF vectors on combined corpus
    corpus = A_texts + B_texts
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(corpus)

    A_vec = X[:len(A_texts)]
    B_vec = X[len(A_texts):]

    # 3) Cosine similarity matrix
    S = cosine_similarity(A_vec, B_vec)  # shape: [len(A) x len(B)]

    # 4) Greedy matching: for each A, pick best B above threshold
    matched_A = set()
    matched_B = set()
    matches: List[Tuple[int, int, float]] = []

    for i in range(S.shape[0]):
        j = int(np.argmax(S[i]))
        sim_ij = float(S[i, j])

        if sim_ij >= sim_threshold:
            matched_A.add(i)
            matched_B.add(j)
            matches.append((i, j, sim_ij))

    A_only = [list_A[i] for i in range(len(list_A)) if i not in matched_A]
    B_only = [list_B[j] for j in range(len(list_B)) if j not in matched_B]

    return {
        "similarity_matrix": S,
        "matches": matches,
        "A_only": A_only,
        "B_only": B_only,
    }

result = compare_critique_lists_tfidf(list_A,list_B, sim_threshold=0.2)

print("matches:", result["matches"])      # aligned issues (indices + similarity)
print("A:", result["A_only"])
print("B:", result["B_only"])


matches: [(0, 0, 0.3860356449047112), (1, 1, 0.6317836962800402), (2, 4, 0.5421647047396755), (5, 2, 0.2809153068884753)]
A: [{'expected_standard': 'Icons that aren’t universally recognized should include text labels or clear contextual cues so users don’t have to infer meaning.', 'observed_issue': 'Right-aligned icons (checkmark, lock, info) are unlabeled. The checkmark’s meaning (selected, started, or completed) is ambiguous, forcing interpretation.', 'suggested_fix': 'Replace the checkmark with a labeled CTA (e.g., "Start"), add a small "Locked"/price label next to the lock, or pair icons with short labels. Ensure consistent icon semantics across the list.', 'guideline_reference': 'Nielsen Norman Heuristics — Recognition rather than recall; Consistency and standards. Apple Human Interface Guidelines — Icons: Use familiar symbols and provide text labels for clarity.'}, {'expected_standard': 'Unavailable or paywalled items should be clearly identified and explain how to gain access to

### token Jaccard similarity

In [ ]:
from typing import List, Dict, Any, Tuple, Set
import re

def _text_to_token_set(text: str) -> Set[str]:
    # lowercase, keep only alphabetic/number tokens
    tokens = re.findall(r"\w+", text.lower())
    return set(tokens)

def _jaccard(set_a: Set[str], set_b: Set[str]) -> float:
    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0
    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    return inter / union if union > 0 else 0.0

def compare_critique_lists_jaccard(
    list_A: List[Dict[str, Any]],
    list_B: List[Dict[str, Any]],
    field: str = "observed_issue",
    sim_threshold: float = 0.3,
):
    """
    Compare two lists of critiques using token-level Jaccard similarity
    over the selected field (default: 'observed_issue').
    """

    A_texts = [c.get(field, "") for c in list_A]
    B_texts = [c.get(field, "") for c in list_B]

    if not A_texts or not B_texts:
        return {
            "similarity_matrix": None,
            "matches": [],
            "A_only": list_A,
            "B_only": list_B,
        }

    A_sets = [_text_to_token_set(t) for t in A_texts]
    B_sets = [_text_to_token_set(t) for t in B_texts]

    # similarity matrix [len(A) x len(B)]
    S = [[_jaccard(A_sets[i], B_sets[j]) for j in range(len(B_sets))]
         for i in range(len(A_sets))]

    matched_A = set()
    matched_B = set()
    matches: List[Tuple[int, int, float]] = []

    for i, row in enumerate(S):
        j = max(range(len(row)), key=lambda k: row[k])
        sim_ij = row[j]
        if sim_ij >= sim_threshold:
            matched_A.add(i)
            matched_B.add(j)
            matches.append((i, j, sim_ij))

    A_only = [list_A[i] for i in range(len(list_A)) if i not in matched_A]
    B_only = [list_B[j] for j in range(len(list_B)) if j not in matched_B]

    return {
        "similarity_matrix": S,
        "matches": matches,
        "A_only": A_only,
        "B_only": B_only,
    }

result = compare_critique_lists_jaccard(list_A,list_B, sim_threshold=0.2)

print("matches:", result["matches"])      # aligned issues (indices + similarity)
print("A:", result["A_only"])
print("B:", result["B_only"])

matches: [(0, 0, 0.3333333333333333), (1, 1, 0.5), (2, 4, 0.48148148148148145), (5, 2, 0.25806451612903225)]
A: [{'expected_standard': 'Icons that aren’t universally recognized should include text labels or clear contextual cues so users don’t have to infer meaning.', 'observed_issue': 'Right-aligned icons (checkmark, lock, info) are unlabeled. The checkmark’s meaning (selected, started, or completed) is ambiguous, forcing interpretation.', 'suggested_fix': 'Replace the checkmark with a labeled CTA (e.g., "Start"), add a small "Locked"/price label next to the lock, or pair icons with short labels. Ensure consistent icon semantics across the list.', 'guideline_reference': 'Nielsen Norman Heuristics — Recognition rather than recall; Consistency and standards. Apple Human Interface Guidelines — Icons: Use familiar symbols and provide text labels for clarity.'}, {'expected_standard': 'Unavailable or paywalled items should be clearly identified and explain how to gain access to prevent dead

### embedding-based comparison

In [ ]:
from typing import List, Dict, Any, Tuple, Callable
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def _critique_to_text(c: Dict[str, Any], fields=("observed_issue",)):
    """Concatenate selected fields into a single text string."""
    parts = [c.get(f, "") for f in fields]
    return " ".join(p.strip() for p in parts if p).strip()

def compare_critique_lists_emb(
    list_A: List[Dict[str, Any]],
    list_B: List[Dict[str, Any]],
    embed_fn: Callable[[List[str]], np.ndarray],
    fields = ("observed_issue",),   # or ("observed_issue", "suggested_fix")
    sim_threshold: float = 0.7,
):
    """
    Compare two lists of critiques using neural embeddings + cosine similarity.

    - list_A, list_B: lists of critique dicts
    - embed_fn: function(List[str]) -> np.ndarray [n x d]
    - fields: which critique fields to include in the text representation
    """

    A_texts = [_critique_to_text(c, fields) for c in list_A]
    B_texts = [_critique_to_text(c, fields) for c in list_B]

    if not A_texts or not B_texts:
        return {
            "similarity_matrix": None,
            "matches": [],
            "A_only": list_A,
            "B_only": list_B,
            "recall": 0.0,
            "precision": 0.0,
            "jaccard": 0.0,
        }

    # 1) Embed
    A_emb = np.asarray(embed_fn(A_texts))
    B_emb = np.asarray(embed_fn(B_texts))

    # 2) Cosine similarity matrix [len(A) x len(B)]
    S = cosine_similarity(A_emb, B_emb)

    # 3) Greedy matching: for each Aᵢ, pick best Bⱼ above threshold
    matched_A = set()
    matched_B = set()
    matches: List[Tuple[int, int, float]] = []

    for i in range(S.shape[0]):
        j = int(np.argmax(S[i]))
        sim_ij = float(S[i, j])
        if sim_ij >= sim_threshold:
            matched_A.add(i)
            matched_B.add(j)
            matches.append((i, j, sim_ij))

    A_only = [list_A[i] for i in range(len(list_A)) if i not in matched_A]
    B_only = [list_B[j] for j in range(len(list_B)) if j not in matched_B]

    # Simple per-screen metrics
    if len(list_A) > 0:
        recall = len(matches) / len(list_A)
    else:
        recall = 0.0

    if len(list_B) > 0:
        precision = len(matches) / len(list_B)
    else:
        precision = 0.0

    denom = len(list_A) + len(list_B) - len(matches)
    jaccard = len(matches) / denom if denom > 0 else 0.0

    return {
        "similarity_matrix": S,
        "matches": matches,   # list of (idx_in_A, idx_in_B, similarity)
        "A_only": A_only,
        "B_only": B_only,
        "recall": recall,
        "precision": precision,
        "jaccard": jaccard,
    }


from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_fn(texts: List[str]):
    return model.encode(texts, convert_to_numpy=True)

result = compare_critique_lists_emb(
    list_A,
    list_B,
    embed_fn=embed_fn,
    fields=("observed_issue",),   # or include "suggested_fix"
    sim_threshold=0.7,
)

result["matches"], result["A_only"], result["B_only"], result["recall"], result["precision"]

Run all methods for a screen

In [ ]:
# from typing import List, Dict, Any, Callable, Optional, Tuple
# import re
# import json
# import numpy as np

# from difflib import SequenceMatcher
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity

# from sentence_transformers import SentenceTransformer
# import tensorflow_hub as hub

# # ---------- generic helpers ----------

# def _normalize_text(s: str) -> str:
#     return re.sub(r"\s+", " ", s.strip().lower())

# def _token_set(s: str):
#     return set(re.findall(r"\w+", s.lower()))

# def _jaccard(a: set, b: set) -> float:
#     if not a and not b:
#         return 1.0
#     if not a or not b:
#         return 0.0
#     inter = len(a & b)
#     union = len(a | b)
#     return inter / union if union else 0.0

# def _sequence_sim(a: str, b: str) -> float:
#     return SequenceMatcher(None, a, b).ratio()

# def _greedy_match(sim: np.ndarray, threshold: float) -> Tuple[List[Tuple[int,int,float]], List[int], List[int]]:
#     """
#     Greedy one-way matching: for each Aᵢ, pick best Bⱼ >= threshold.
#     Returns (matches, A_unmatched_idx, B_unmatched_idx).
#     """
#     matched_A = set()
#     matched_B = set()
#     matches: List[Tuple[int,int,float]] = []

#     for i in range(sim.shape[0]):
#         j = int(np.argmax(sim[i]))
#         score = float(sim[i, j])
#         if score >= threshold:
#             matched_A.add(i)
#             matched_B.add(j)
#             matches.append((i, j, round(score, 3)))

#     A_only = [i for i in range(sim.shape[0]) if i not in matched_A]
#     B_only = [j for j in range(sim.shape[1]) if j not in matched_B]
#     return matches, A_only, B_only

# def _coverage_metrics(nA: int, nB: int, n_matches: int) -> Dict[str, float]:
#     recall = n_matches / nA if nA > 0 else 0.0
#     precision = n_matches / nB if nB > 0 else 0.0
#     denom = nA + nB - n_matches
#     jaccard = n_matches / denom if denom > 0 else 0.0
#     return {"recall": recall, "precision": precision, "jaccard": jaccard}


# # ---------- method-specific similarity matrices ----------

# def _sim_exact(A: List[str], B: List[str]) -> np.ndarray:
#     A_norm = [_normalize_text(x) for x in A]
#     B_norm = [_normalize_text(x) for x in B]
#     sim = np.zeros((len(A), len(B)), dtype=float)
#     for i, a in enumerate(A_norm):
#         for j, b in enumerate(B_norm):
#             sim[i, j] = 1.0 if a == b else 0.0
#     return sim

# def _sim_seq(A: List[str], B: List[str]) -> np.ndarray:
#     sim = np.zeros((len(A), len(B)), dtype=float)
#     for i, a in enumerate(A):
#         for j, b in enumerate(B):
#             sim[i, j] = _sequence_sim(a, b)
#     return sim

# def _sim_jaccard(A: List[str], B: List[str]) -> np.ndarray:
#     A_sets = [_token_set(x) for x in A]
#     B_sets = [_token_set(x) for x in B]
#     sim = np.zeros((len(A), len(B)), dtype=float)
#     for i, sa in enumerate(A_sets):
#         for j, sb in enumerate(B_sets):
#             sim[i, j] = _jaccard(sa, sb)
#     return sim

# def _sim_tfidf(A: List[str], B: List[str]) -> np.ndarray:
#     corpus = A + B
#     if not corpus:
#         return np.zeros((len(A), len(B)), dtype=float)
#     vec = TfidfVectorizer()
#     X = vec.fit_transform(corpus)
#     A_vec = X[:len(A)]
#     B_vec = X[len(A):]
#     return cosine_similarity(A_vec, B_vec)

# def _sim_embed(A: List[str], B: List[str], embed_fn: Callable[[List[str]], np.ndarray]) -> np.ndarray:
#     if not A or not B:
#         return np.zeros((len(A), len(B)), dtype=float)
#     A_emb = np.asarray(embed_fn(A))
#     B_emb = np.asarray(embed_fn(B))
#     return cosine_similarity(A_emb, B_emb)

# def load_embed_fn(model_key, embed_models, client=None):
#     name = embed_models[model_key]

#     # --- SentenceTransformers ---
#     if model_key.startswith("st_"):
#         model = SentenceTransformer(name)
#         return lambda texts: model.encode(texts, convert_to_numpy=True)

#     # --- OpenAI ---
#     if model_key.startswith("openai_"):
#         assert client is not None, "OpenAI client must be provided."
#         return lambda texts: [
#             item.embedding for item in client.embeddings.create(model=name, input=texts).data
#         ]

#     # --- Universal Sentence Encoder ---
#     if model_key == "use":
#         model = hub.load(name)
#         return lambda texts: model(texts).numpy()

#     raise ValueError(f"Unknown model key: {model_key}")

# # ---------- constants ----------
# SIM_FUNCS = {
#     "exact":    _sim_exact,
#     "sequence": _sim_seq,
#     "jaccard":  _sim_jaccard,
#     "tfidf":    _sim_tfidf,
#     "embedding": _sim_embed,   # special handling below
# }

# # Similarity methods that don't need embeddings
# BASELINE_METHODS = ["exact", "sequence", "jaccard", "tfidf"]

# # Embedding model registry
# EMBED_MODELS = {
#     # SentenceTransformers
#     "st_minilm":        "all-MiniLM-L6-v2",             # fast, reliable baseline
#     "st_mpnet":         "all-mpnet-base-v2",            # strongest general embedding model
#     "st_mpnet_qa":      "multi-qa-mpnet-base-dot-v1",   # optimized for similarity search        
#     "st_distilroberta": "all-distilroberta-v1",         # smaller, still good

#     # OpenAI
#     "openai_large":     "text-embedding-3-large",
#     "openai_small":     "text-embedding-3-small",

#     # Universal Sentence Encoder
#     "use":              "https://tfhub.dev/google/universal-sentence-encoder/4",
# }

# # ---------- main driver ----------
# def compare_observed_list(
#     observed_A,
#     observed_B,
#     method: str,
#     threshold: float = 0.5,
#     embed_fn=None
# ):
#     """
#     Compare two lists of observed_issue strings using ONE method.
#     method ∈ {"exact", "sequence", "jaccard", "tfidf", "embedding"}
#     """

#     # ---------- select similarity function ----------
#     if threshold is None:
#         threshold = SIM_THRESHOLDS[method]

#     if method not in SIM_FUNCS:
#         raise ValueError(f"Unknown method: {method}")

#     # dispatch
#     if method == "embedding":
#         if embed_fn is None:
#             raise ValueError("embedding method requires embed_fn")
#         S = SIM_FUNCS[method](observed_A, observed_B, embed_fn)
#     else:
#         S = SIM_FUNCS[method](observed_A, observed_B)

#     # ---------- greedy matching ----------
#     matches, A_only_idx, B_only_idx = _greedy_match(S, threshold)
#     metrics = _coverage_metrics(len(observed_A), len(observed_B), len(matches))

#     return {
#         "method": method,
#         "similarity_matrix": S,
#         "matches": matches,
#         "A_only_idx": A_only_idx,
#         "B_only_idx": B_only_idx,
#         **metrics,
#     }

# def summarize_comparisons(
#     observed_A,
#     observed_B,
#     thresholds,
#     client=None,
# ):
#     """
#     Summarize comparisons between observed_A and observed_C
#     using multiple methods (baseline + embedding-based).
#     Returns a DataFrame with one row per method.
#     """
#     rows = []

#     # --- 1) Baseline methods (no embeddings) ---
#     for method in BASELINE_METHODS:
#         res = compare_observed_list(
#             observed_A=observed_A,
#             observed_B=observed_B,
#             method=method,
#             threshold=thresholds[method],
#             embed_fn=None,
#         )

#         rows.append({
#             "method": method,
#             "matches": [(i, j) for (i, j, _s) in res["matches"]],
#             "A_only": res["A_only_idx"],
#             "B_only": res["B_only_idx"],
#         })
        
#     # --- 2) Embedding-based methods (one row per embedding model) ---
#     for model_key in EMBED_MODELS.keys():
#         embed_fn = load_embed_fn(model_key, EMBED_MODELS, client=client)
#         res = compare_observed_list(
#             observed_A=observed_A,
#             observed_B=observed_B,
#             method="embedding",
#             threshold=thresholds["embedding"],
#             embed_fn=embed_fn,
#         )

#         rows.append({
#             "method": f"embedding_{model_key}",
#             "matches": [(i, j) for (i, j, _s) in res["matches"]],
#             "A_only": res["A_only_idx"],
#             "B_only": res["B_only_idx"],
#         })

#     return pd.DataFrame(rows).set_index("method")

thresholds = {
    "exact": 0.999,
    "sequence": 0.60,
    "jaccard": 0.30,
    "tfidf": 0.30,
    "embedding": 0.5,
}

summary_df = summarize_comparisons(observed_A, observed_C, thresholds, client)
summary_df

,matches,A_only,B_only
method,,,
exact,[],"[0, 1, 2, 3, 4, 5]","[0, 1, 2, 3, 4, 5, 6]"
sequence,"[(0, 0), (1, 1), (5, 4)]","[2, 3, 4]","[2, 3, 5, 6]"
jaccard,"[(0, 0), (1, 1), (2, 2), (5, 4)]","[3, 4]","[3, 5, 6]"
tfidf,"[(0, 0), (1, 1), (2, 2), (3, 5), (5, 4)]",[4],"[3, 6]"
embedding_st_minilm,"[(0, 0), (1, 1), (2, 2), (3, 3), (4, 6), (5, 4)]",[],[5]
embedding_st_mpnet,"[(0, 0), (1, 1), (2, 2), (3, 3), (4, 6), (5, 4)]",[],[5]
embedding_st_mpnet_qa,"[(0, 0), (1, 1), (2, 2), (3, 5), (4, 6), (5, 4)]",[],[3]
embedding_st_distilroberta,"[(0, 0), (1, 1), (2, 2), (3, 5), (5, 4)]",[4],"[3, 6]"
embedding_openai_large,"[(0, 0), (1, 1), (2, 2), (3, 3), (4, 6), (5, 4)]",[],[5]


### All methods

In [56]:
from typing import List, Dict, Any, Tuple, Callable, Optional
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# ----------------- text extraction -----------------

def _critique_to_text(c: Dict[str, Any], fields=("observed_issue",)) -> str:
    parts = []
    for f in fields:
        v = c.get(f, "")
        if v is None:
            continue
        v = str(v).strip()
        if v:
            parts.append(v)
    return " | ".join(parts).strip()

def _texts_with_index_map(items: List[Dict[str, Any]], fields) -> Tuple[List[str], List[int]]:
    """
    Returns:
      texts: only non-empty texts
      idx_map: original indices for each returned text
    """
    texts, idx_map = [], []
    for i, c in enumerate(items):
        t = _critique_to_text(c, fields)
        if t:
            texts.append(t)
            idx_map.append(i)
    return texts, idx_map


# ----------------- matching + metrics -----------------

def _greedy_one_to_one_maxmatch(S: np.ndarray, threshold: float) -> List[Tuple[int, int, float]]:
    """
    One-to-one greedy matching (global):
      - consider all pairs (i,j) with S[i,j] >= threshold
      - sort pairs by similarity desc
      - take pair if neither i nor j already matched
    Returns list of (i, j, sim)
    """
    if S.size == 0:
        return []

    pairs = np.argwhere(S >= threshold)
    if pairs.size == 0:
        return []

    sims = S[pairs[:, 0], pairs[:, 1]]
    order = np.argsort(-sims)  # descending

    matched_i = set()
    matched_j = set()
    matches: List[Tuple[int, int, float]] = []

    for k in order:
        i = int(pairs[k, 0])
        j = int(pairs[k, 1])
        if i in matched_i or j in matched_j:
            continue
        sim = float(S[i, j])
        matches.append((i, j, sim))
        matched_i.add(i)
        matched_j.add(j)

    return matches

def _coverage_metrics(nA: int, nB: int, n_matches: int) -> Dict[str, float]:
    # interpret A as "reference" (e.g., expert), B as "candidate" (e.g., model)
    recall = n_matches / nA if nA else 0.0
    precision = n_matches / nB if nB else 0.0
    denom = nA + nB - n_matches
    jaccard = n_matches / denom if denom else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return {"recall": recall, "precision": precision, "f1": f1, "jaccard": jaccard}


# ----------------- main function (embedding-only) -----------------

def compare_critique_lists_emb(
    list_A: List[Dict[str, Any]],
    list_B: List[Dict[str, Any]],
    embed_fn: Callable[[List[str]], np.ndarray],
    fields=("observed_issue",),          # or ("observed_issue","suggested_fix")
    sim_threshold: float = 0.7,
    return_matrix: bool = False,
) -> Dict[str, Any]:
    """
    Embedding-based issue agreement between list_A and list_B.

    Convention:
      - list_A = reference (usually expert)
      - list_B = candidate (usually model)
      => recall = matched / len(A), precision = matched / len(B)
    """

    A_texts, A_map = _texts_with_index_map(list_A, fields)
    B_texts, B_map = _texts_with_index_map(list_B, fields)

    # handle empties after filtering blanks
    if not A_texts or not B_texts:
        nA, nB = len(A_texts), len(B_texts)
        m = 0
        out = {
            "matches": [],
            "A_only_idx": A_map,  # all non-empty are unmatched
            "B_only_idx": B_map,
            "nA": nA, "nB": nB, "n_matches": m,
            **_coverage_metrics(nA, nB, m),
            "mean_match_sim": None,
        }
        if return_matrix:
            out["similarity_matrix"] = None
        return out

    # embed
    A_emb = np.asarray(embed_fn(A_texts))
    B_emb = np.asarray(embed_fn(B_texts))

    # cosine similarity matrix
    S = cosine_similarity(A_emb, B_emb)

    # one-to-one matching in filtered-text index space
    matches_local = _greedy_one_to_one_maxmatch(S, sim_threshold)

    # convert matches back to ORIGINAL indices
    matches = [(A_map[i], B_map[j], sim) for (i, j, sim) in matches_local]

    matched_A = {i for (i, _, _) in matches}
    matched_B = {j for (_, j, _) in matches}

    A_only_idx = [i for i in A_map if i not in matched_A]
    B_only_idx = [j for j in B_map if j not in matched_B]

    nA, nB, m = len(A_texts), len(B_texts), len(matches)
    sims = [s for (_, _, s) in matches]
    out = {
        "matches": matches,          # (orig_idx_in_A, orig_idx_in_B, sim)
        "A_only_idx": A_only_idx,
        "B_only_idx": B_only_idx,
        "nA": nA, "nB": nB, "n_matches": m,
        **_coverage_metrics(nA, nB, m),
        "mean_match_sim": float(np.mean(sims)) if sims else None,
        "median_match_sim": float(np.median(sims)) if sims else None,
    }
    if return_matrix:
        out["similarity_matrix"] = S
        out["A_texts"] = A_texts
        out["B_texts"] = B_texts
        out["A_map"] = A_map
        out["B_map"] = B_map
    return out

In [94]:
import pandas as pd

def issue_agreement_over_df(
    responses_df: pd.DataFrame,
    embed_fn: Callable[[List[str]], np.ndarray],
    fields=("observed_issue",),
    sim_threshold=0.7,
    expert_comment_type_value="Human",
) -> pd.DataFrame:
    rows = []
    for _, r in responses_df.iterrows():
        model_list = r.get("critiques") or []

        expert_all = r.get("parsed_comments"); expert_all = [] if expert_all is None else list(expert_all)
        
        expert_list = [c for c in expert_all if str(c.get("source","")).lower() == "human"]

        res = compare_critique_lists_emb(
            list_A=expert_list,
            list_B=model_list,
            embed_fn=embed_fn,
            fields=fields,
            sim_threshold=sim_threshold,
            return_matrix=False,
        )

        rows.append({
            "screen_task_id": r.get("screen_task_id"),
            "screen_id": r.get("screen_id"),
            "n_expert": res["nA"],
            "n_model": res["nB"],
            "n_matches": res["n_matches"],
            "recall": res["recall"],
            "precision": res["precision"],
            "f1": res["f1"],
            "jaccard": res["jaccard"],
            "mean_match_sim": res["mean_match_sim"],
            "median_match_sim": res.get("median_match_sim"),

        })

    return pd.DataFrame(rows)

In [95]:
test_df = responses_df.head(10)

In [107]:
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import numpy as np

# # ---------- constants ----------
# SIM_FUNCS = {
#     "exact":    _sim_exact,
#     "sequence": _sim_seq,
#     "jaccard":  _sim_jaccard,
#     "tfidf":    _sim_tfidf,
#     "embedding": _sim_embed,   # special handling below
# }

# # Similarity methods that don't need embeddings
# BASELINE_METHODS = ["exact", "sequence", "jaccard", "tfidf"]

# # Embedding model registry
# EMBED_MODELS = {
#     # SentenceTransformers
#     "st_minilm":        "all-MiniLM-L6-v2",             # fast, reliable baseline
#     "st_mpnet":         "all-mpnet-base-v2",            # strongest general embedding model
#     "st_mpnet_qa":      "multi-qa-mpnet-base-dot-v1",   # optimized for similarity search        
#     "st_distilroberta": "all-distilroberta-v1",         # smaller, still good

#     # OpenAI
#     "openai_large":     "text-embedding-3-large",
#     "openai_small":     "text-embedding-3-small",

#     # Universal Sentence Encoder
#     "use":              "https://tfhub.dev/google/universal-sentence-encoder/4",
# }

client = OpenAI()
openai_embed_fn = lambda texts: np.array(
    [d.embedding for d in client.embeddings.create(model="text-embedding-3-small", input=texts).data],
    dtype=np.float32
)

st_model = SentenceTransformer("all-mpnet-base-v2")
st_embed_fn = lambda texts: st_model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)


screen_metrics_df = issue_agreement_over_df(test_df, embed_fn=st_embed_fn, fields=("observed_issue",), sim_threshold=0.45)

In [ ]:
screen_metrics_df

,screen_task_id,screen_id,n_expert,n_model,n_matches,recall,precision,f1,jaccard,mean_match_sim,median_match_sim
0,15_T01,15,9,10,4,0.444444,0.400000,0.421053,0.266667,0.527451,0.492506
1,28_T01,28,0,8,0,0.000000,0.000000,0.000000,0.000000,NaN,NaN
2,67_T01,67,7,3,1,0.142857,0.333333,0.200000,0.111111,0.530270,0.530270
3,190_T01,190,3,4,0,0.000000,0.000000,0.000000,0.000000,NaN,NaN
4,193_T01,193,4,7,1,0.250000,0.142857,0.181818,0.100000,0.593260,0.593260
5,233_T01,233,0,6,0,0.000000,0.000000,0.000000,0.000000,NaN,NaN
6,288_T01,288,2,5,1,0.500000,0.200000,0.285714,0.166667,0.572139,0.572139
7,342_T01,342,4,6,0,0.000000,0.000000,0.000000,0.000000,NaN,NaN
8,422_T01,422,3,4,1,0.333333,0.250000,0.285714,0.166667,0.545035,0.545035
9,445_T01,445,5,8,1,0.200000,0.125000,0.153846,0.083333,0.573189,0.573189


In [105]:
screen_metrics_df


,screen_task_id,screen_id,n_expert,n_model,n_matches,recall,precision,f1,jaccard,mean_match_sim,median_match_sim
0,15_T01,15,9,10,1,0.111111,0.100000,0.105263,0.055556,0.647274,0.647274
1,28_T01,28,0,8,0,0.000000,0.000000,0.000000,0.000000,NaN,NaN
2,67_T01,67,7,3,1,0.142857,0.333333,0.200000,0.111111,0.530270,0.530270
3,190_T01,190,3,4,0,0.000000,0.000000,0.000000,0.000000,NaN,NaN
4,193_T01,193,4,7,1,0.250000,0.142857,0.181818,0.100000,0.593260,0.593260
5,233_T01,233,0,6,0,0.000000,0.000000,0.000000,0.000000,NaN,NaN
6,288_T01,288,2,5,1,0.500000,0.200000,0.285714,0.166667,0.572139,0.572139
7,342_T01,342,4,6,0,0.000000,0.000000,0.000000,0.000000,NaN,NaN
8,422_T01,422,3,4,1,0.333333,0.250000,0.285714,0.166667,0.545035,0.545035
9,445_T01,445,5,8,1,0.200000,0.125000,0.153846,0.083333,0.573189,0.573189


In [112]:
screen_task_id = "15_T01"
r = responses_df.loc[responses_df["screen_task_id"] == screen_task_id].iloc[0]

model_list = list(r["critiques"] or [])
expert_all = list(r["parsed_comments"] or [])
expert_list = [c for c in expert_all if str(c.get("source","")).lower() == "human"]

res = compare_critique_lists_emb(
    list_A=expert_list,   # expert = reference
    list_B=model_list,    # model = candidate
    embed_fn=openai_embed_fn,
    fields=("expected_standard","observed_issue"),
    sim_threshold=0.45,
    return_matrix=False
)

print("MATCHES (expert_idx, model_idx, sim):")
for a_i, b_j, s in sorted(res["matches"], key=lambda x: -x[2]):
    print(f"\nSIM={s:.3f}")
    print("  EXP:", expert_list[a_i]["observed_issue"])
    print("  LLM:", model_list[b_j]["observed_issue"])

print("\nEXPERT-ONLY:")
for i in res["A_only_idx"]:
    print("-", expert_list[i]["observed_issue"])

print("\nLLM-ONLY:")
for j in res["B_only_idx"]:
    print("-", model_list[j]["observed_issue"])


MATCHES (expert_idx, model_idx, sim):

SIM=0.664
  EXP: text (Plans) is in light red color on white background  which is not making a good contrast.
  LLM: Light orange text (e.g., section header Get Started and secondary descriptions) on white appears low-contrast, reducing readability.

SIM=0.517
  EXP: Icon is not clearly visible
  LLM: The trailing info icon (action) and the checkmark (status) use similar color and placement, making it harder to distinguish action vs. status at a glance.

SIM=0.470
  EXP: the back button size is small.
  LLM: The small info and lock/check icons on the right appear tightly spaced and likely below the recommended touch target size.

EXPERT-ONLY:
- the text (Workouts) is small even when it is heading.
- the texts are disappearing at the bottom edge of the layout leaving no marginal space which is making it difficult for users to know the complete information
- "Plus" sign overlaps the other elements (Icon)
- there is no button on the page for navigati

In [118]:
from typing import List, Dict, Any, Tuple
import json
import pandas as pd
from openai import OpenAI

client = OpenAI()

# ----------------- core: one call per screen -----------------

def llm_match_critiques_pairs(
    expert_list: List[Dict[str, Any]],
    model_list:  List[Dict[str, Any]],
    fields: Tuple[str, ...] = ("observed_issue",),
    model_name: str = "gpt-4o-mini",
) -> List[List[int]]:
    """
    Returns ONLY a JSON array of pairs: [[expert_i, model_j], ...]
    One-to-one matching is required.
    """

    def to_text(c: Dict[str, Any]) -> str:
        parts = []
        for f in fields:
            v = c.get(f, "")
            if v is None:
                continue
            v = str(v).strip()
            if v:
                parts.append(v)
        return " | ".join(parts).strip()

    A = [to_text(c) for c in expert_list]
    B = [to_text(c) for c in model_list]

    # guard
    if not A or not B:
        return []

    prompt = (
    "Task: Match expert critique issues (A) with model critique issues (B).\n\n"

    "Two items should match if they describe the same underlying UI/usability problem, "
    "even if phrased differently.\n"
    "They do not need to use the same wording, but they should refer to the same UI element "
    "and the same core issue.\n\n"

    "Do not match items that are only loosely related or that describe different problems, "
    "even if they occur in the same area of the screen.\n\n"

    "Constraints:\n"
    "- One-to-one matching (each index may be used at most once).\n"
    "- It is acceptable to return few or zero matches if appropriate.\n\n"

    "Output format:\n"
    "Return ONLY a JSON array of pairs [[expert_index, model_index], ...].\n"
    "No explanations. No extra text. No markdown.\n\n"

    "A items:\n" +
    "\n".join([f"{i}. {t}" for i, t in enumerate(A)]) +
    "\n\nB items:\n" +
    "\n".join([f"{j}. {t}" for j, t in enumerate(B)])
)


    resp = client.responses.create(
        model=model_name,
        input=prompt,
    )

    # Parse JSON array [[i,j],...]
    text = (resp.output_text or "").strip()

    try:
        pairs = json.loads(text)
    except json.JSONDecodeError:
        # fallback: try to extract JSON substring if the model added stray text
        start = text.find("[")
        end = text.rfind("]")
        if start == -1 or end == -1 or end <= start:
            return []
        pairs = json.loads(text[start:end+1])

    # sanitize: ensure list of [int,int], unique indices (defensive)
    clean = []
    used_a = set()
    used_b = set()
    for p in pairs:
        if not (isinstance(p, list) and len(p) == 2):
            continue
        a, b = p
        if not (isinstance(a, int) and isinstance(b, int)):
            continue
        if a < 0 or b < 0 or a >= len(A) or b >= len(B):
            continue
        if a in used_a or b in used_b:
            continue
        used_a.add(a)
        used_b.add(b)
        clean.append([a, b])

    return clean


# ----------------- metrics helper -----------------

def _metrics(nA: int, nB: int, n_matches: int) -> Dict[str, float]:
    recall = n_matches / nA if nA else 0.0
    precision = n_matches / nB if nB else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    denom = nA + nB - n_matches
    jaccard = n_matches / denom if denom else 0.0
    return {"recall": recall, "precision": precision, "f1": f1, "jaccard": jaccard}


# ----------------- dataframe driver -----------------

def new_issue_agreement_over_df_llm(
    responses_df: pd.DataFrame,
    fields: Tuple[str, ...] = ("observed_issue",),
    model_name: str = "gpt-4o-mini",
    expert_source_value: str = "human",   # your data: c["source"] == "Human"
    keep_pairs: bool = True,
) -> pd.DataFrame:
    rows = []

    for _, r in responses_df.iterrows():
        model_list = r.get("critiques")
        model_list = [] if model_list is None else list(model_list)

        expert_all = r.get("parsed_comments")
        expert_all = [] if expert_all is None else list(expert_all)

        # Human only
        expert_list = [c for c in expert_all if str(c.get("source", "")).lower() == expert_source_value.lower()]

        pairs = llm_match_critiques_pairs(
            expert_list=expert_list,
            model_list=model_list,
            fields=fields,
            model_name=model_name,
        )

        nA, nB, nm = len(expert_list), len(model_list), len(pairs)
        m = _metrics(nA, nB, nm)

        out = {
            "screen_task_id": r.get("screen_task_id"),
            "screen_id": r.get("screen_id"),
            "n_expert": nA,
            "n_model": nB,
            "n_matches": nm,
            **m,
        }

        if keep_pairs:
            out["matches_pairs"] = pairs
            out["expert_only_idx"] = [i for i in range(nA) if i not in {a for a, _b in pairs}]
            out["model_only_idx"]  = [j for j in range(nB) if j not in {_a for _a, b in pairs}]

        rows.append(out)

    return pd.DataFrame(rows)


# ----------------- example usage -----------------

screen_metrics_df = new_issue_agreement_over_df_llm(
    responses_df=test_df,
    fields=("observed_issue",),  # or ("expected_standard","observed_issue","suggested_fix")
    model_name="gpt-4o-mini",
)
screen_metrics_df

,screen_task_id,screen_id,n_expert,n_model,n_matches,recall,precision,f1,jaccard,matches_pairs,expert_only_idx,model_only_idx
0,15_T01,15,9,10,7,0.777778,0.700000,0.736842,0.583333,"[[0, 5], [2, 3], [3, 6], [4, 2], [6, 0], [7, 1...","[1, 5]","[1, 5, 9]"
1,28_T01,28,0,8,0,0.000000,0.000000,0.000000,0.000000,[],[],"[0, 1, 2, 3, 4, 5, 6, 7]"
2,67_T01,67,7,3,2,0.285714,0.666667,0.400000,0.250000,"[[3, 1], [0, 2]]","[1, 2, 4, 5, 6]","[1, 2]"
3,190_T01,190,3,4,3,1.000000,0.750000,0.857143,0.750000,"[[0, 0], [1, 1], [2, 2]]",[],[3]
4,193_T01,193,4,7,4,1.000000,0.571429,0.727273,0.571429,"[[0, 1], [1, 2], [2, 3], [3, 0]]",[],"[4, 5, 6]"
5,233_T01,233,0,6,0,0.000000,0.000000,0.000000,0.000000,[],[],"[0, 1, 2, 3, 4, 5]"
6,288_T01,288,2,5,2,1.000000,0.400000,0.571429,0.400000,"[[0, 0], [1, 2]]",[],"[2, 3, 4]"
7,342_T01,342,4,6,4,1.000000,0.666667,0.800000,0.666667,"[[0, 2], [1, 3], [2, 1], [3, 4]]",[],"[4, 5]"
8,422_T01,422,3,4,2,0.666667,0.500000,0.571429,0.400000,"[[0, 0], [1, 3]]",[2],"[2, 3]"
9,445_T01,445,5,8,5,1.000000,0.625000,0.769231,0.625000,"[[0, 4], [1, 6], [2, 7], [3, 1], [4, 0]]",[],"[5, 6, 7]"


In [115]:
screen_metrics_df.loc[0, "matches_pairs"]  # example: list of matched index pairs for first screen

[[1, 5], [2, 3], [3, 6], [4, 0], [6, 1], [7, 4], [8, 2]]

In [119]:
screen_metrics_df.loc[0, "matches_pairs"]  # example: list of matched index pairs for first screen


[[0, 5], [2, 3], [3, 6], [4, 2], [6, 0], [7, 1], [8, 8]]

In [116]:
screen_task_id = "15_T01"

# get row
r = responses_df.loc[responses_df["screen_task_id"] == screen_task_id].iloc[0]

model_list = [] if r["critiques"] is None else list(r["critiques"])
expert_all = [] if r["parsed_comments"] is None else list(r["parsed_comments"])

# human only
expert_list = [c for c in expert_all if str(c.get("source","")).lower() == "human"]

# get stored pairs
pairs = screen_metrics_df.loc[
    screen_metrics_df["screen_task_id"] == screen_task_id,
    "matches_pairs"
].iloc[0]

print(f"\nMatches for {screen_task_id}:\n")

for a, b in pairs:
    print(f"[Expert {a}]")
    print(" ", expert_list[a]["observed_issue"])
    print(f"[Model  {b}]")
    print(" ", model_list[b]["observed_issue"])
    print("-" * 80)



Matches for 15_T01:

[Expert 1]
  text (Plans) is in light red color on white background  which is not making a good contrast.
[Model  5]
  Light orange text (e.g., section header Get Started and secondary descriptions) on white appears low-contrast, reducing readability.
--------------------------------------------------------------------------------
[Expert 2]
  the back button size is small.
[Model  3]
  The small info and lock/check icons on the right appear tightly spaced and likely below the recommended touch target size.
--------------------------------------------------------------------------------
[Expert 3]
  the texts are disappearing at the bottom edge of the layout leaving no marginal space which is making it difficult for users to know the complete information
[Model  6]
  Workout rows lack disclosure indicators or buttons, leaving it unclear whether tapping a row opens details or starts a workout.
------------------------------------------------------------------------

In [122]:
i = 0
r = test_df.iloc[i]

model_list = r["critiques"]; model_list = [] if model_list is None else list(model_list)
expert_all = r["parsed_comments"]; expert_all = [] if expert_all is None else list(expert_all)

expert_list = [c for c in expert_all if str(c.get("comment_id","")).lower().startswith("comment")]

print("EXPERT observed_issue samples:")
for k, c in enumerate(expert_list[:], 1):
    print(k - 1, repr(c.get("observed_issue","")))

print("\nMODEL observed_issue samples:")
for k, c in enumerate(model_list[:], 1):
    print(k-1, repr(c.get("observed_issue","")))


EXPERT observed_issue samples:
0 'the text (Workouts) is small even when it is heading.'
1 'text (Plans) is in light red color on white background  which is not making a good contrast.'
2 'the back button size is small.'
3 'the texts are disappearing at the bottom edge of the layout leaving no marginal space which is making it difficult for users to know the complete information'
4 'Icon is not clearly visible'
5 '"Plus" sign overlaps the other elements (Icon)'
6 'there is no button on the page for navigating further.'
7 'the text is difficult to read because the font is too small.'
8 'the elements (Plus SIgn) on the page are not positioned in a way that creates a sense of balance or movement.'

MODEL observed_issue samples:
0 'The Full Body item shows only a checkmark and an info icon; there is no obvious Start action, making it unclear how to begin the workout.'
1 'The trailing checkmark beside Full Body lacks a label, making it unclear whether it indicates selection, completion, or 

In [124]:
# Combine all parsed_comments from the same screen_id across all tasks
# and attach them to responses_df as `extended_parsed_comments`

import pandas as pd

# 1️⃣ Aggregate all expert comments per screen_id from full dataset
screen_to_all_comments = (
    experts_df
    .groupby("screen_id")["parsed_comments"]
    .apply(lambda lists: [
        c
        for sublist in lists if sublist is not None
        for c in list(sublist)
    ])
    .to_dict()
)

# 2️⃣ Map them back into responses_df
responses_df["extended_parsed_comments"] = responses_df["screen_id"].map(
    screen_to_all_comments
)

# 3️⃣ Optional: ensure no None
responses_df["extended_parsed_comments"] = responses_df["extended_parsed_comments"].apply(
    lambda x: [] if x is None else x
)


# Store Results